# PoliMillionaire bounded ReAct math solver

This notebook uses a local language model as a bounded ReAct-style math agent.

The default model is now `meta-llama/Llama-3.1-8B-Instruct`, loaded in 4-bit for Colab T4. Paste your Hugging Face token into the model-loading cell before running it.

The runtime still respects the 30-second question limit:

- 4-bit loading is enabled by default on GPU
- each question gets at most one short tool-planning call by default
- deterministic SymPy tools handle exact arithmetic and one-variable equations

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import json
import math
import textwrap
from getpass import getpass

import torch
import sympy as sp

In [ ]:
BASE_DIR_CANDIDATES = [
    '/content/gdrive/MyDrive/NLP_assignment1',
    '/content/gdrive/MyDrive/NLP_assignment',
    '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment',
]
BASE_DIR = next((candidate for candidate in BASE_DIR_CANDIDATES if os.path.exists(candidate)), BASE_DIR_CANDIDATES[0])
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR:', BASE_DIR)
print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('Could not find the NLP assignment folder in Google Drive. Update BASE_DIR_CANDIDATES.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

print('Path added successfully.')

In [ ]:
%pip install -q --upgrade "transformers==4.51.3" "accelerate==1.3.0" "huggingface_hub>=0.30.0,<1.0" sentencepiece protobuf sympy "bitsandbytes>=0.46.1,<0.49"

import sys
import importlib.metadata as metadata

print('transformers:', metadata.version('transformers'))
print('accelerate:', metadata.version('accelerate'))
print('huggingface_hub:', metadata.version('huggingface_hub'))

if 'transformers' in sys.modules:
    print('Note: transformers was already imported. If model loading fails, restart the kernel manually and run from the top.')

In [ ]:
import importlib.metadata as metadata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

print('Using transformers:', metadata.version('transformers'))

In [ ]:
API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
USERNAME = (__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip())
PASSWORD = (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip())

if PASSWORD is None:
    PASSWORD = getpass('Millionaire password: ')

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

In [ ]:
competitions = client.competitions.list_all()
for competition in competitions:
    print(competition.id, competition.name, competition.max_levels)

COMPETITION_ID = 3
print('Selected competition:', COMPETITION_ID)

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = (__import__("os").getenv("HF_TOKEN") or __import__("getpass").getpass("Hugging Face token (input hidden): ").strip())
if HF_TOKEN == 'PASTE_YOUR_HF_TOKEN_HERE' or not HF_TOKEN.strip():
    raise ValueError('Paste your Hugging Face token into HF_TOKEN before loading Llama.')

login(token=HF_TOKEN, add_to_git_credential=False)

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
MAX_PROMPT_TOKENS = int(os.getenv('MILLIONAIRE_MAX_PROMPT_TOKENS', '1024'))
DEFAULT_MAX_NEW_TOKENS = int(os.getenv('MILLIONAIRE_MAX_NEW_TOKENS', '96'))
OFFLOAD_DIR = os.getenv('MILLIONAIRE_OFFLOAD_DIR', '/content/llama_offload')
os.makedirs(OFFLOAD_DIR, exist_ok=True)
DEFAULT_TEMPERATURE = float(os.getenv('MILLIONAIRE_TEMPERATURE', '0.2'))
DEFAULT_TOP_P = float(os.getenv('MILLIONAIRE_TOP_P', '0.9'))

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA GPU not detected. Llama 8B 4-bit is intended for a GPU runtime such as Colab T4.')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading model:', model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_memory = {0: os.getenv('MILLIONAIRE_GPU_MAX_MEMORY', '13GiB'), 'cpu': os.getenv('MILLIONAIRE_CPU_MAX_MEMORY', '24GiB')}

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map='auto',
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR,
    offload_state_dict=True,
    offload_buffers=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)
model.eval()
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = True


def make_prompt(user_message):
    messages = [{'role': 'user', 'content': user_message}]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return user_message


def generate_text(user_message, max_new_tokens=None, temperature=None):
    max_new_tokens = DEFAULT_MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens
    temperature = DEFAULT_TEMPERATURE if temperature is None else temperature
    prompt = make_prompt(user_message)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    ).to(model.device)
    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'pad_token_id': tokenizer.eos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'do_sample': temperature > 0,
        'use_cache': True,
    }
    if temperature > 0:
        generation_kwargs['temperature'] = temperature
        generation_kwargs['top_p'] = DEFAULT_TOP_P

    with torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
SAFE_LOCALS = {
    'sqrt': sp.sqrt,
    'root': sp.root,
    'sin': sp.sin,
    'cos': sp.cos,
    'tan': sp.tan,
    'asin': sp.asin,
    'acos': sp.acos,
    'atan': sp.atan,
    'log': sp.log,
    'ln': sp.log,
    'exp': sp.exp,
    'factorial': sp.factorial,
    'binomial': sp.binomial,
    'floor': sp.floor,
    'ceil': sp.ceiling,
    'ceiling': sp.ceiling,
    'Abs': sp.Abs,
    'abs': sp.Abs,
    'pi': sp.pi,
    'E': sp.E,
}


def clean_math_text(text):
    text = str(text).strip()
    replacements = {
        '^': '**',
        chr(8722): '-',
        chr(215): '*',
        chr(8901): '*',
        chr(247): '/',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = text.replace('{', '(').replace('}', ')')
    return text


def validate_math_text(text, allow_equals=False):
    if '\\' in text or '__' in text or '[' in text or ']' in text:
        raise ValueError('Unsupported math syntax.')
    allowed = r'[0-9A-Za-z_+\-*/().,%\s]+'
    if allow_equals:
        allowed = r'[0-9A-Za-z_+\-*/().,%=\s]+'
    if not re.fullmatch(allowed, text):
        raise ValueError('Expression contains unsupported characters.')


def calculator(expression):
    expression = clean_math_text(expression)
    validate_math_text(expression, allow_equals=False)
    value = sp.simplify(sp.sympify(expression, locals=SAFE_LOCALS))
    if value.free_symbols:
        raise ValueError('Expression still contains variables. Use equation_solver for equations.')

    result = {
        'tool': 'calculator',
        'input': expression,
        'exact': str(value),
        'decimal': str(sp.N(value, 12)),
    }
    if value.is_integer:
        result['integer'] = str(int(value))
    return result


def equation_solver(equation, variable='x'):
    variable = str(variable).strip()
    if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]*', variable):
        raise ValueError('Invalid variable name.')

    equation = clean_math_text(equation)
    validate_math_text(equation, allow_equals=True)
    if equation.count('=') != 1:
        raise ValueError('Equation must contain exactly one equals sign.')

    symbol = sp.Symbol(variable)
    locals_with_variable = dict(SAFE_LOCALS)
    locals_with_variable[variable] = symbol
    left, right = equation.split('=', 1)
    solutions = sp.solve(
        sp.Eq(sp.sympify(left, locals=locals_with_variable), sp.sympify(right, locals=locals_with_variable)),
        symbol,
    )

    return {
        'tool': 'equation_solver',
        'equation': equation,
        'variable': variable,
        'solutions': [str(sp.simplify(solution)) for solution in solutions],
        'decimal_solutions': [str(sp.N(solution, 12)) for solution in solutions],
    }


def call_tool(action, action_input):
    action = str(action).strip().lower()
    try:
        if action == 'calculator':
            if isinstance(action_input, dict):
                action_input = action_input.get('expression', '')
            return {'ok': True, 'result': calculator(action_input)}

        if action == 'equation_solver':
            if isinstance(action_input, dict):
                equation = action_input.get('equation', '')
                variable = action_input.get('variable', 'x')
            else:
                equation = str(action_input)
                variable = 'x'
            return {'ok': True, 'result': equation_solver(equation, variable)}

        return {'ok': False, 'error': f'Unknown tool: {action}'}
    except Exception as exc:
        return {'ok': False, 'error': f'{type(exc).__name__}: {exc}'}


def extract_json_objects(text):
    text = str(text).strip()
    fence = chr(96) * 3
    if text.startswith(fence):
        text = text[len(fence):].strip()
        if text.lower().startswith('json'):
            text = text[4:].strip()
    if text.endswith(fence):
        text = text[:-len(fence)].strip()

    decoder = json.JSONDecoder()
    objects = []
    for index, char in enumerate(text):
        if char != '{':
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            objects.append(value)
    return objects


def extract_last_json_object(text):
    objects = extract_json_objects(text)
    return objects[-1] if objects else None


def extract_letter(text):
    text = str(text).strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    if match:
        return match.group(1)
    return text[0] if text and text[0] in 'ABCD' else None


def question_to_text(question):
    lines = [str(question.text).strip()]
    for index, option in enumerate(question.options[:4]):
        lines.append(f'{chr(65 + index)}) {option.text}')
    return '\n'.join(lines)


def option_id_for_letter(question, letter):
    index = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[index].id

In [ ]:
MODEL_TEMPERATURE = float(os.getenv('MILLIONAIRE_TEMPERATURE', '0.2'))
STRUCTURED_TOKENS = int(os.getenv('MILLIONAIRE_STRUCTURED_TOKENS', '192'))
LETTER_TOKENS = int(os.getenv('MILLIONAIRE_LETTER_TOKENS', '8'))
USE_STRUCTURED_LLM = os.getenv('MILLIONAIRE_USE_STRUCTURED_LLM', '1') != '0'

LETTERS = ['A', 'B', 'C', 'D']


def normalize_text(value):
    text = str(value).lower()
    text = text.replace(chr(8722), '-')
    text = text.replace(chr(8211), '-')
    text = text.replace(chr(8212), '-')
    text = text.replace('?', '<=')
    text = text.replace('?', '>=')
    text = text.replace('$', '')
    return re.sub(r'\s+', ' ', text).strip()


def make_trace(source, letter, thought, observation=None):
    return [{'step': 0, 'action': {'action': source, 'answer': letter, 'thought': thought}, 'observation': observation or {}, 'raw_output': 'deterministic generic rule'}]


def make_result(letter, confidence, reason, source, observation=None):
    return {'ok': True, 'answer': letter, 'confidence': confidence, 'reason': reason, 'source': source, 'trace': make_trace(source, letter, reason, observation)}


def option_texts(question):
    return {LETTERS[i]: str(option.text) for i, option in enumerate(question.options[:4])}


def option_letter_matching(question, predicate):
    for letter, text in option_texts(question).items():
        if predicate(normalize_text(text)):
            return letter
    return None


def option_letter_containing(question, phrase):
    phrase = normalize_text(phrase)
    return option_letter_matching(question, lambda text: phrase in text)


def option_id_for_letter_safe(question, letter):
    return option_id_for_letter(question, letter if letter in LETTERS else 'A')


def numeric_option_values(question):
    values = {}
    for letter, text in option_texts(question).items():
        match = re.search(r'[-+]?\d+(?:\.\d+)?', text.replace(',', ''))
        if match:
            values[letter] = float(match.group(0))
    return values


def parse_option_math_value(text):
    expr = normalize_text(text)
    expr = expr.replace('?', 'pi')
    expr = expr.replace('^', '**')
    expr = re.sub(r'(\d)\s*sqrt', r'\1*sqrt', expr)
    expr = re.sub(r'(\d)\s*pi', r'\1*pi', expr)
    expr = expr.replace(' ', '')
    if not re.fullmatch(r'[0-9.+\-*/()sqrtpi]+', expr):
        return None
    try:
        value = sp.N(sp.sympify(expr, locals={'sqrt': sp.sqrt, 'pi': sp.pi}), 12)
    except Exception:
        return None
    if getattr(value, 'free_symbols', set()):
        return None
    return float(value)


def math_option_values(question):
    values = {}
    for letter, text in option_texts(question).items():
        value = parse_option_math_value(text)
        if value is not None:
            values[letter] = value
    return values


def closest_option_by_value(question, target):
    values = math_option_values(question) or numeric_option_values(question)
    if not values:
        return None
    return min(values, key=lambda letter: abs(values[letter] - float(target)))


def option_letter_for_integer(question, target):
    for letter, text in option_texts(question).items():
        match = re.fullmatch(r'\s*(-?\d+)\s*', text)
        if match and int(match.group(1)) == int(target):
            return letter
    return closest_option_by_value(question, target)


def word_to_number(text):
    numbers = {'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19, 'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50, 'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90}
    return numbers.get(normalize_text(text))


def parse_number_word_or_digits(text):
    text = normalize_text(text)
    return int(text) if re.fullmatch(r'\d+', text) else word_to_number(text)


def is_prime_integer(value):
    value = int(value)
    if value < 2:
        return False
    if value == 2:
        return True
    if value % 2 == 0:
        return False
    for divisor in range(3, int(math.sqrt(value)) + 1, 2):
        if value % divisor == 0:
            return False
    return True


def first_primes_at_least(start, count):
    primes = []
    candidate = int(start)
    while len(primes) < int(count):
        if is_prime_integer(candidate):
            primes.append(candidate)
        candidate += 1
    return primes


def sum_digits(value):
    return sum(int(char) for char in str(abs(int(value))))


def binomial_tail_probability(n, p, at_least):
    return sum(math.comb(n, k) * (p ** k) * ((1 - p) ** (n - k)) for k in range(at_least, n + 1))


def stirling_second_kind(n, k):
    if n == k == 0:
        return 1
    if n == 0 or k == 0 or k > n:
        return 0
    table = [[0] * (k + 1) for _ in range(n + 1)]
    table[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, min(i, k) + 1):
            table[i][j] = j * table[i - 1][j] + table[i - 1][j - 1]
    return table[n][k]


def parse_linear_coefficients(expression):
    expr = normalize_text(expression).replace(' ', '')
    coeffs = {'x': 0.0, 'y': 0.0, 'z': 0.0}
    for sign, coeff, var in re.findall(r'([+-]?)(\d*(?:\.\d+)?)?([xyz])', expr):
        magnitude = 1.0 if coeff in {'', None} else float(coeff)
        if sign == '-':
            magnitude = -magnitude
        coeffs[var] += magnitude
    return coeffs


def extract_sets_from_question(text):
    compact = text.replace(' ', '')
    sets = {}
    for label, contents in re.findall(r'([ivx]+):\{([^}]*)\}', compact):
        values = [float(part) for part in contents.split(',') if part]
        sets[label.upper()] = values
    return sets


def population_sd(values):
    if not values:
        return 0.0
    mean = sum(values) / len(values)
    return math.sqrt(sum((value - mean) ** 2 for value in values) / len(values))


def option_letter_for_roman_pair(question, first, second):
    target = f'{first},{second}'.lower().replace(' ', '')
    return option_letter_matching(question, lambda text: text.replace(' ', '') == target)


def option_letter_for_truth_pair(question, first_truth, second_truth):
    target = f'{str(first_truth)}, {str(second_truth)}'.lower().replace(' ', '')
    return option_letter_matching(question, lambda text: text.replace(' ', '') == target)


def generic_rule_answer(question):
    qtext = normalize_text(question.text)
    full_text = normalize_text(question_to_text(question))

    congruence = re.search(r'from\s+(-?\d+)\s+to\s+(-?\d+).*?congruent\s+to\s+(-?\d+)\s+(?:pmod|mod|modulo)\s*\{?(-?\d+)\}?', full_text)
    if congruence:
        low, high, residue, modulus = map(int, congruence.groups())
        if modulus != 0:
            first = low + ((residue - low) % abs(modulus))
            count = 0 if first > high else ((high - first) // abs(modulus)) + 1
            letter = option_letter_for_integer(question, count)
            if letter:
                return make_result(letter, 'high', f'Count integers x in [{low},{high}] with x congruent to {residue} mod {abs(modulus)}: {count}.', 'rule:modular_interval_count', {'low': low, 'high': high, 'residue': residue, 'modulus': abs(modulus), 'count': count})

    if 'type ii error' in qtext:
        letter = option_letter_matching(question, lambda opt: ('continue to use' in opt or 'keep' in opt) and ('actually more effective' in opt or 'in excess' in opt))
        if letter:
            return make_result(letter, 'high', 'A Type II error fails to reject a false null hypothesis.', 'rule:hypothesis_type_ii', {'error': 'fail to reject false H0'})
    if 'type i error' in qtext:
        letter = option_letter_matching(question, lambda opt: ('closing' in opt or 'prescribe' in opt or 'begin' in opt) and ('within the allowed limit' in opt or 'not any more effective' in opt))
        if letter:
            return make_result(letter, 'high', 'A Type I error rejects a true null hypothesis.', 'rule:hypothesis_type_i', {'error': 'reject true H0'})

    if 'least likely to reduce bias' in qtext and 'sample survey' in qtext:
        letter = option_letter_containing(question, 'stratified random sampling')
        if letter:
            return make_result(letter, 'high', 'Random sampling, neutral wording, and nonresponse follow-up directly reduce bias; stratification mainly improves representation/precision.', 'rule:survey_bias_methods', {})

    if 'percentile' in qtext:
        percentile_values = [float(value) for value in re.findall(r'(\d+(?:\.\d+)?)(?:st|nd|rd|th)', qtext)]
        if len(percentile_values) >= 2:
            diff = abs(percentile_values[0] - percentile_values[1])
            target = int(round(diff))
            letter = None
            for opt_letter, opt_text in option_texts(question).items():
                opt_norm = normalize_text(opt_text)
                digit = re.search(r'(\d+(?:\.\d+)?)\s*%', opt_norm)
                word = re.search(r'\b([a-z]+)\s+percent\b', opt_norm)
                if 'between' in opt_norm and ((digit and abs(float(digit.group(1)) - diff) < 1e-9) or (word and word_to_number(word.group(1)) == target)):
                    letter = opt_letter
                    break
            if letter:
                return make_result(letter, 'high', f'The percent between two percentile ranks is their difference: {diff:g}%.', 'rule:percentile_difference', {'percentiles': percentile_values[:2], 'difference': diff})

    quartile_binomial = ('quartile' in qtext and 'with replacement' in qtext and 'at least' in qtext)
    if quartile_binomial:
        sample = re.search(r'(\d+|one|two|three|four|five|six|seven|eight|nine|ten)\s+[^.]*selected at random', qtext)
        at_least = re.search(r'at least\s+(\d+|one|two|three|four|five|six|seven|eight|nine|ten)', qtext)
        if sample and at_least:
            n = parse_number_word_or_digits(sample.group(1))
            k = parse_number_word_or_digits(at_least.group(1))
            p = 0.25 if 'third quartile' in qtext or 'first quartile' in qtext else None
            if n is not None and k is not None and p is not None:
                probability = binomial_tail_probability(n, p, k)
                letter = closest_option_by_value(question, probability)
                if letter:
                    return make_result(letter, 'high', f'Quartile threshold gives p={p}; P(Binomial({n},{p})>={k})={probability:.4f}.', 'rule:quartile_binomial_tail', {'n': n, 'p': p, 'at_least': k, 'probability': probability})

    if 'selling price' in qtext and 'age' in qtext:
        slope_match = re.search(r'selling price\s*=\s*[-+]?\d+(?:\.\d+)?\s*[-+]\s*(\d+(?:\.\d+)?)\s*\(?age\)?', qtext)
        if slope_match and ('1,000' in qtext or '1000' in qtext):
            dollars = round(float(slope_match.group(1)) * 1000)
            letter = option_letter_matching(question, lambda opt: ('drop' in opt or 'down' in opt or 'decrease' in opt) and str(dollars) in opt.replace(',', ''))
            if letter:
                return make_result(letter, 'high', f'The slope is -{slope_match.group(1)} thousand dollars per year, about {dollars} dollars per year.', 'rule:regression_slope_units', {'slope_thousands': -float(slope_match.group(1)), 'dollars': dollars})

    linear_ball = re.search(r'(minimum|maximum) value of (?:the expression )?(.+?) as .*?subject to .*?x\^2 \+ y\^2 \+ z\^2\s*<=\s*(\d+(?:\.\d+)?)', qtext)
    if linear_ball:
        mode, expr, radius_sq_text = linear_ball.groups()
        coeffs = parse_linear_coefficients(expr)
        target = math.sqrt(sum(value ** 2 for value in coeffs.values())) * math.sqrt(float(radius_sq_text)) * (-1 if mode == 'minimum' else 1)
        letter = closest_option_by_value(question, target)
        if letter:
            return make_result(letter, 'high', f'For a linear function on a ball, the {mode} is {target:g} by Cauchy-Schwarz.', 'rule:linear_objective_on_ball', {'coefficients': coeffs, 'radius_squared': float(radius_sq_text), 'target': target})

    xy_match = re.search(r'shortest distance from (?:the )?curve\s*x\s*y\s*=\s*([0-9]+(?:\.[0-9]+)?)\s*to (?:the )?origin', qtext)
    if xy_match:
        k_value = float(xy_match.group(1))
        distance = math.sqrt(2 * abs(k_value))
        letter = closest_option_by_value(question, distance)
        if letter:
            return make_result(letter, 'high', f'With xy={k_value:g}, distance squared is minimized at x^2+y^2=2|xy|={2*abs(k_value):g}.', 'rule:distance_xy_to_origin', {'k': k_value, 'distance': distance})

    if 'first half' in qtext and 'second half' in qtext and ('mph' in qtext or 'miles per hour' in qtext):
        distance_match = re.search(r'(\d+(?:\.\d+)?)\s*[- ]?mile', qtext)
        speeds = [float(value) for value in re.findall(r'(?:at|of)\s+(\d+(?:\.\d+)?)\s*(?:mph|miles per hour)', qtext)]
        if distance_match and len(speeds) >= 2:
            distance = float(distance_match.group(1))
            total_time = (distance / 2) / speeds[0] + (distance / 2) / speeds[1]
            letter = closest_option_by_value(question, total_time)
            if letter:
                return make_result(letter, 'high', f'Total time is half-distance over each speed: {total_time:.3f} hours.', 'rule:two_speed_halves', {'distance': distance, 'speeds': speeds[:2], 'total_time': total_time})

    prime_digit = re.search(r'product of the two smallest\s+(\d+)\s*[- ]?digit prime', qtext)
    if prime_digit and 'sum of the digits' in qtext:
        digits = int(prime_digit.group(1))
        primes = first_primes_at_least(10 ** (digits - 1), 2)
        product = math.prod(primes)
        digit_sum = sum_digits(product)
        letter = option_letter_for_integer(question, digit_sum)
        if letter:
            return make_result(letter, 'high', f'The two smallest {digits}-digit primes are {primes[0]} and {primes[1]}; product {product} has digit sum {digit_sum}.', 'rule:prime_product_digit_sum', {'primes': primes, 'product': product, 'digit_sum': digit_sum})

    compact = qtext.replace(' ', '')
    field_match = re.search(r'q\(sqrt\((\d+)\)\+sqrt\((\d+)\)\).*overq', compact)
    if field_match:
        a, b = map(int, field_match.groups())
        root = math.isqrt(a * b)
        degree = 2 if a == b or root * root == a * b else 4
        letter = option_letter_for_integer(question, degree)
        if letter:
            return make_result(letter, 'high', f'Q(sqrt({a})+sqrt({b})) has degree {degree} over Q.', 'rule:quadratic_radical_field_degree', {'a': a, 'b': b, 'degree': degree})

    balls = re.search(r'(\d+|one|two|three|four|five|six|seven|eight|nine|ten)\s+distinguishable balls? into\s+(\d+|one|two|three|four|five|six|seven|eight|nine|ten)\s+indistinguishable boxes?', qtext)
    if balls:
        n = parse_number_word_or_digits(balls.group(1))
        k = parse_number_word_or_digits(balls.group(2))
        if n is not None and k is not None:
            nonempty = 'nonempty' in qtext or 'no box empty' in qtext
            ways = stirling_second_kind(n, k) if nonempty else sum(stirling_second_kind(n, used) for used in range(1, min(k, n) + 1))
            letter = option_letter_for_integer(question, ways)
            if letter:
                return make_result(letter, 'high', f'Distributing {n} distinguishable balls into {k} indistinguishable boxes gives {ways} ways.', 'rule:distinguishable_balls_indistinguishable_boxes', {'n': n, 'k': k, 'nonempty': nonempty, 'ways': ways})

    if 'standard deviation' in qtext and ('total combined' in qtext or 'random variable w' in qtext):
        sd_match = re.search(r'standard deviation of (?:about )?(\d+(?:\.\d+)?)', qtext)
        n_match = re.search(r'if\s+(\d+|one|two|three|four|five|six|seven|eight|nine|ten)\s+[^.]*selected', qtext)
        if sd_match and n_match:
            sd_each = float(sd_match.group(1))
            n = parse_number_word_or_digits(n_match.group(1))
            sd_sum = math.sqrt(n) * sd_each
            letter = closest_option_by_value(question, sd_sum)
            if letter:
                return make_result(letter, 'high', f'SD of a sum of {n} independent variables is sqrt({n}) times {sd_each:g}, or {sd_sum:g}.', 'rule:sd_sum_independent', {'n': n, 'sd_each': sd_each, 'sd_sum': sd_sum})

    if 'smallest standard deviation' in qtext and 'largest' in qtext:
        sets = extract_sets_from_question(qtext)
        if len(sets) >= 2:
            sds = {label: population_sd(values) for label, values in sets.items()}
            smallest = min(sds, key=sds.get)
            largest = max(sds, key=sds.get)
            letter = option_letter_for_roman_pair(question, smallest, largest)
            if letter:
                return make_result(letter, 'high', f'{smallest} has the smallest SD and {largest} has the largest SD.', 'rule:compare_set_standard_deviations', {'standard_deviations': sds})

    if 'homomorphism' in qtext and 'kernel consists of the identity' in qtext:
        image_order = re.search(r'image of a group of\s+(\d+)\s+elements.*?may have\s+(\d+)\s+elements', qtext)
        if image_order:
            domain_order, candidate_order = map(int, image_order.groups())
            first_truth = True
            second_truth = domain_order % candidate_order == 0
            letter = option_letter_for_truth_pair(question, first_truth, second_truth)
            if letter:
                return make_result(letter, 'high', 'Trivial kernel iff injective; image order must divide domain order.', 'rule:group_homomorphism_kernel_image_order', {'statement_1': first_truth, 'statement_2': second_truth})
    if 'homomorphic image' in qtext and 'cyclic' in qtext and 'abelian' in qtext:
        letter = option_letter_for_truth_pair(question, True, True)
        if letter:
            return make_result(letter, 'high', 'Homomorphic images preserve cyclic generation and commutativity.', 'rule:homomorphic_image_properties', {'statement_1': True, 'statement_2': True})

    if 'correct description of the term' in qtext and 'replication means' in full_text:
        letter = option_letter_containing(question, 'replication means')
        if letter:
            return make_result(letter, 'medium', 'Replication is the only listed description that matches the experimental-design term.', 'rule:experiment_design_vocabulary', {})

    return None


def build_structured_answer_prompt(question):
    return f'''Answer this multiple-choice question. Use step-by-step reasoning internally and eliminate wrong choices internally.
Return exactly one valid JSON object and nothing else. Do not use Markdown. Do not reveal chain-of-thought.

JSON schema:
{{"answer":"A","confidence":"low|medium|high","reason":"one concise sentence","eliminated":{{"A":"short note","B":"short note","C":"short note","D":"short note"}}}}

Question:
{question_to_text(question)}

JSON:'''


def run_structured_answer(question):
    raw_output = generate_text(build_structured_answer_prompt(question), max_new_tokens=STRUCTURED_TOKENS, temperature=MODEL_TEMPERATURE)
    result = extract_last_json_object(raw_output)
    letter = extract_letter(result.get('answer', '')) if isinstance(result, dict) else None
    if letter not in LETTERS:
        return {'ok': False, 'answer': None, 'confidence': 'low', 'reason': 'Structured LLM did not return valid JSON.', 'raw_output': raw_output}
    return {'ok': True, 'answer': letter, 'confidence': str(result.get('confidence', 'medium')).strip().lower(), 'reason': str(result.get('reason', 'structured answer')).strip(), 'eliminated': result.get('eliminated', {}), 'raw_output': raw_output, 'source': 'llm:structured_answer'}


def build_letter_only_prompt(question):
    return f'''Return only one character: A, B, C, or D. Do not explain.

{question_to_text(question)}

Answer:'''


def extract_strict_letter_output(raw_output):
    text = str(raw_output).strip().upper()
    if '<THINK>' in text or '</THINK>' in text:
        return None
    match = re.fullmatch(r'([ABCD])\s*[).:]*', text)
    return match.group(1) if match else None


def run_letter_only_fallback(question):
    raw_output = generate_text(build_letter_only_prompt(question), max_new_tokens=LETTER_TOKENS, temperature=0.0)
    letter = extract_strict_letter_output(raw_output)
    if letter not in LETTERS:
        return None
    return {'ok': True, 'answer': letter, 'confidence': 'medium', 'reason': 'Selected by strict letter-only fallback.', 'raw_output': raw_output, 'source': 'llm:letter_only'}


def build_letter_score_prompt(question):
    return f'''Answer the question with only A, B, C, or D.

{question_to_text(question)}

Answer:'''


def score_option_letters(question):
    prompt = make_prompt(build_letter_score_prompt(question))
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_PROMPT_TOKENS).to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1]
    scores = {}
    for letter in LETTERS:
        candidate_scores = []
        for candidate in [letter, ' ' + letter, '\n' + letter]:
            token_ids = tokenizer.encode(candidate, add_special_tokens=False)
            if token_ids:
                candidate_scores.append(float(logits[token_ids[-1]].detach().cpu()))
        scores[letter] = max(candidate_scores) if candidate_scores else float('-inf')
    return scores


def run_letter_score_fallback(question):
    scores = score_option_letters(question)
    letter = max(scores, key=scores.get)
    return {'ok': True, 'answer': letter, 'confidence': 'low', 'reason': 'Selected by direct A/B/C/D token scoring.', 'scores': {key: round(value, 4) for key, value in scores.items()}, 'source': 'llm:letter_score'}


def choose_answer(question):
    started = time.monotonic()
    if len(question.options) < 4:
        return question.options[0].id, 'A', {'ok': False, 'reason': 'fewer than four options'}

    rule_result = generic_rule_answer(question)
    if rule_result:
        raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_rule': True, 'used_fallback': False, 'final_result': rule_result, 'react_result': rule_result}
        return option_id_for_letter_safe(question, rule_result['answer']), rule_result['answer'], raw

    structured_result = {'ok': False, 'answer': None, 'reason': 'structured LLM skipped'}
    if USE_STRUCTURED_LLM:
        structured_result = run_structured_answer(question)
        if structured_result.get('ok'):
            trace_result = {'ok': True, 'reason': structured_result.get('reason'), 'trace': [{'step': 0, 'action': {'action': 'structured_answer', 'answer': structured_result.get('answer'), 'confidence': structured_result.get('confidence'), 'eliminated': structured_result.get('eliminated', {})}, 'raw_output': structured_result.get('raw_output', '')}]}
            raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_rule': False, 'used_fallback': False, 'structured_result': structured_result, 'final_result': structured_result, 'react_result': trace_result}
            return option_id_for_letter_safe(question, structured_result['answer']), structured_result['answer'], raw

    fallback_result = run_letter_only_fallback(question)
    used_fallback = True
    if not fallback_result:
        fallback_result = run_letter_score_fallback(question)

    letter = fallback_result.get('answer') if fallback_result else 'A'
    if letter not in LETTERS:
        letter = 'A'
        fallback_result = {'ok': False, 'answer': letter, 'confidence': 'low', 'reason': 'No valid answer after all fallbacks.'}

    trace_result = {'ok': bool(fallback_result.get('ok')), 'reason': fallback_result.get('reason'), 'trace': [{'step': 0, 'action': {'action': fallback_result.get('source', 'fallback'), 'answer': letter, 'confidence': fallback_result.get('confidence')}, 'raw_output': fallback_result.get('raw_output', ''), 'observation': {'structured_result': structured_result}}]}
    raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_rule': False, 'used_fallback': used_fallback, 'structured_result': structured_result, 'final_result': fallback_result, 'react_result': trace_result}
    return option_id_for_letter_safe(question, letter), letter, raw

In [ ]:
TRACE_WIDTH = int(os.getenv('MILLIONAIRE_TRACE_WIDTH', '100'))
TRACE_JSON_CHARS = int(os.getenv('MILLIONAIRE_TRACE_JSON_CHARS', '900'))


def compact_json(value, max_chars=TRACE_JSON_CHARS):
    try:
        text = json.dumps(value, ensure_ascii=True, indent=2)
    except TypeError:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + ' ...[truncated]'
    return text


def wrapped(text, width=TRACE_WIDTH, indent=''):
    text = str(text).strip()
    if not text:
        return ''
    lines = []
    for line in text.splitlines():
        if len(line) <= width:
            lines.append(indent + line)
        else:
            lines.append(textwrap.fill(line, width=width, initial_indent=indent, subsequent_indent=indent))
    return '\n'.join(lines)


def print_block(title, value=None):
    print(title, flush=True)
    if value is not None:
        print(wrapped(value, indent='  '), flush=True)


def print_agent_trace(raw, show_raw_model_output=False):
    print('', flush=True)
    print('================ Agent trace ================', flush=True)
    print('Elapsed seconds:', raw.get('elapsed_seconds'), flush=True)
    print('Used rule:', raw.get('used_rule'), flush=True)
    print('Used fallback:', raw.get('used_fallback'), flush=True)

    react_result = raw.get('react_result', {})
    print('React status:', react_result.get('ok'), '| reason:', react_result.get('reason'), flush=True)

    trace = react_result.get('trace', [])
    if not trace:
        print('No ReAct trace was recorded.', flush=True)

    for step in trace:
        print('---------------------------------------------', flush=True)
        print('Step:', step.get('step'), flush=True)
        if step.get('error'):
            print('Error:', step.get('error'), flush=True)

        action = step.get('action')
        if action:
            display_action = {key: value for key, value in action.items() if not key.startswith('_')}
            print_block('Model action:', compact_json(display_action))

        if step.get('observation') is not None:
            print_block('Tool observation:', compact_json(step.get('observation')))

        if show_raw_model_output and step.get('raw_output'):
            print_block('Raw model output:', step.get('raw_output'))

    final = raw.get('final_result', {})
    print('---------------------------------------------', flush=True)
    print_block('Final result object:', compact_json(final))
    print('============== End agent trace ==============', flush=True)
    print('', flush=True)


def play_game(
    competition_id=COMPETITION_ID,
    max_questions=None,
    delay=0.2,
    show_trace=True,
    show_raw_model_output=False,
):
    game = client.game.start(competition_id=competition_id, mode='text')
    answered = 0

    while game.in_progress:
        if max_questions is not None and answered >= max_questions:
            break

        question = game.current_question
        if question is None:
            break

        print('', flush=True)
        print('=============================================', flush=True)
        print('Level:', game.current_level, flush=True)
        print(wrapped(question.text), flush=True)
        for index, option in enumerate(question.options):
            print(wrapped(f'{chr(65 + index)}) {option.text}'), flush=True)

        option_id, letter, raw = choose_answer(question)
        final = raw.get('final_result', {})
        print('Predicted:', letter, '| confidence:', final.get('confidence'), '| reason:', final.get('reason'), flush=True)

        if show_trace:
            print_agent_trace(raw, show_raw_model_output=show_raw_model_output)

        try:
            result = game.answer(option_id)
        except TimeoutError:
            print('Timed out', flush=True)
            break
        except RateLimitError:
            print('Rate limited, waiting...', flush=True)
            time.sleep(5)
            result = game.answer(option_id)

        answered += 1
        print('Correct:', result.correct, '| Earned:', result.earned_amount, flush=True)

        if result.game_over:
            break

        time.sleep(delay)

    print('Final earned:', game.earned_amount, flush=True)
    return game.earned_amount

In [ ]:
play_game(COMPETITION_ID, show_trace=True, show_raw_model_output=True)